# Câncer de Mama — Análise Exploratória de Dados (EDA)

Este notebook realiza uma análise exploratória completa do dataset `Breast_Cancer.csv`, cobrindo:
- Visão geral e estrutura do dataset
- Valores ausentes e tipos de dados
- Análise univariada (variáveis numéricas e categóricas)
- Análise bivariada e multivariada
- Análise de correlação
- Distribuição da variável-alvo (`Status`)

## 1. Importações e Configurações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.gridspec import GridSpec

# Estilo dos gráficos
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 14

import warnings
warnings.filterwarnings('ignore')

## 2. Carregamento dos Dados

In [ ]:
df = pd.read_csv('base/Breast_Cancer.csv')

# Normaliza nomes das colunas: remove espaços extras
df.columns = df.columns.str.strip()

print(f'Dimensões: {df.shape[0]} linhas x {df.shape[1]} colunas')
df.head()

## 3. Visão Geral do Dataset

In [ ]:
df.info()

Estatísticas descritivas de todas as colunas

In [ ]:
df.describe(include='all').T

## 4. Valores Ausentes

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Qtd. Ausentes': missing,
    '% Ausentes': missing_pct
}).query('`Qtd. Ausentes` > 0')

if missing_df.empty:
    print('Nenhum valor ausente encontrado.')
else:
    display(missing_df)
    fig, ax = plt.subplots()
    missing_df['% Ausentes'].plot(kind='bar', ax=ax, color='tomato')
    ax.set_title('Valores Ausentes (%)')
    ax.set_ylabel('%')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 5. Variável-Alvo: `Status`

Entender o balanceamento das classes para checar possível desbalanceamento significativo entre Alive e Dead

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = df['Status'].value_counts()
counts.plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Status — Contagem')
axes[0].set_ylabel('Contagem')
axes[0].set_xticklabels(counts.index, rotation=0)

axes[1].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Status — Proporção')

plt.suptitle('Distribuição da Variável-Alvo', fontsize=15)
plt.tight_layout()
plt.show()

print(counts.to_string())

## 6. Variáveis Numéricas

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
print('Colunas numéricas:', num_cols)

### 6.1 Distribuições (Histogramas + KDE)

Checar se o formato de cada distribuição numérica é aproximadamente normal, assimétrica ou multimodal.

In [ ]:
n = len(num_cols)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='steelblue')
    axes[i].set_title(col)
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Variáveis Numéricas — Distribuições', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

### 6.2 Boxplots por Status

Checar se a mediana e a dispersão de cada variável numérica diferem entre pacientes Alive e Dead (sinal de relevância preditiva).

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    sns.boxplot(data=df, x='Status', y=col, ax=axes[i],
                palette={'Alive': 'steelblue', 'Dead': 'tomato'})
    axes[i].set_title(col)
    axes[i].set_xlabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Variáveis Numéricas — Boxplot por Status', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

### 6.3 Resumo de Outliers (método IQR)

Utiliza a regra do IQR para sinalizar possíveis outliers. Taxas elevadas em variáveis clínicas (contagem de linfonodos, tamanho do tumor) são comuns e podem refletir casos extremos reais, não erros de dados.

In [ ]:
outlier_summary = []
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_out = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    outlier_summary.append({'Coluna': col, 'Outliers': n_out,
                             'Outliers %': round(n_out / len(df) * 100, 2)})

pd.DataFrame(outlier_summary).sort_values('Outliers', ascending=False)

## 7. Variáveis Categóricas

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.drop('Status').tolist()
print('Colunas categóricas:', cat_cols)

### 7.1 Contagem de Valores

Distribuição de frequência de cada variável categórica. Categorias raras podem precisar ser agrupadas para evitar divisões esparsas durante a modelagem.

In [ ]:
n = len(cat_cols)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, y=col, order=order, ax=axes[i], palette='Set2')
    axes[i].set_title(col)
    axes[i].set_xlabel('Contagem')
    axes[i].set_ylabel('')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Variáveis Categóricas — Contagem de Valores', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

### 7.2 Taxa de Mortalidade por Categoria

Taxa de mortalidade por categoria de cada variável, comparada à média geral (linha tracejada). Categorias bem acima dessa linha são fortes candidatas a preditores de óbito.

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    rate = (df.groupby(col)['Status']
              .apply(lambda x: (x == 'Dead').mean() * 100)
              .sort_values(ascending=False))
    rate.plot(kind='barh', ax=axes[i], color='tomato', edgecolor='black')
    axes[i].set_title(f'Taxa de Óbito (%) por {col}')
    axes[i].set_xlabel('%')
    axes[i].set_ylabel('')
    axes[i].axvline(x=df['Status'].eq('Dead').mean() * 100,
                    color='navy', linestyle='--', label='Média geral')
    axes[i].legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Taxa de Óbito por Variável Categórica', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

## 8. Análise de Correlação

O triângulo inferior da matriz de correlação evita redundância visual. Correlações fortes entre variáveis (|r| > 0,7) podem indicar multicolinearidade, o que merece atenção em modelos lineares.

In [ ]:
corr = df[num_cols].corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Matriz de Correlação — Variáveis Numéricas')
plt.tight_layout()
plt.show()

## 9. Análise dos Meses de Sobrevivência

Os meses de sobrevivência são um proxy essencial do desfecho clínico. A comparação das curvas KDE evidencia como sua distribuição se desloca entre os grupos, enquanto o scatter plot verifica possível interação com a idade do paciente.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE por Status
for status, color in zip(['Alive', 'Dead'], ['steelblue', 'tomato']):
    subset = df[df['Status'] == status]['Survival Months']
    sns.kdeplot(subset, ax=axes[0], label=status, color=color, fill=True, alpha=0.4)
axes[0].set_title('Distribuição dos Meses de Sobrevivência por Status')
axes[0].set_xlabel('Meses de Sobrevivência')
axes[0].legend()

# Dispersão: Meses de Sobrevivência vs Idade
sns.scatterplot(data=df, x='Age', y='Survival Months', hue='Status',
                palette={'Alive': 'steelblue', 'Dead': 'tomato'},
                alpha=0.5, ax=axes[1])
axes[1].set_title('Idade vs Meses de Sobrevivência')

plt.tight_layout()
plt.show()

## 10. Tamanho do Tumor vs Envolvimento Nodal

O estadiamento clínico (T e N) é um dos fatores prognósticos mais importantes no câncer de mama. Estes gráficos verificam se estágios mais avançados correspondem a tumores maiores e maior envolvimento de linfonodos, e se esse padrão difere entre os grupos de sobrevivência.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x='T Stage', y='Tumor Size', hue='Status',
            palette={'Alive': 'steelblue', 'Dead': 'tomato'}, ax=axes[0])
axes[0].set_title('Tamanho do Tumor por T Stage e Status')
axes[0].set_xlabel('T Stage')

sns.boxplot(data=df, x='N Stage', y='Reginol Node Positive', hue='Status',
            palette={'Alive': 'steelblue', 'Dead': 'tomato'}, ax=axes[1])
axes[1].set_title('Linfonodos Positivos por N Stage e Status')
axes[1].set_xlabel('N Stage')

plt.tight_layout()
plt.show()

## 11. Status dos Receptores Hormonais

O status dos receptores de Estrogênio e Progesterona são marcadores clínicos bem estabelecidos. Receptor positivo geralmente indica melhor prognóstico e elegibilidade para terapia hormonal.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, col in zip(axes, ['Estrogen Status', 'Progesterone Status']):
    ct = pd.crosstab(df[col], df['Status'], normalize='index') * 100
    ct.plot(kind='bar', ax=ax, color=['steelblue', 'tomato'], edgecolor='black')
    ax.set_title(f'{col} vs Status (%)')
    ax.set_ylabel('%')
    ax.set_xlabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.legend(title='Status')

plt.suptitle('Status dos Receptores Hormonais vs Sobrevivência', fontsize=14)
plt.tight_layout()
plt.show()

## 12. Pairplot — Variáveis Numéricas por Status

O pairplot oferece uma visão geral das relações par a par e da separação entre classes em todas as variáveis numéricas simultaneamente. As curvas KDE na diagonal mostram a distribuição de cada variável por grupo.

In [ ]:
pp_cols = num_cols + ['Status']
g = sns.pairplot(df[pp_cols], hue='Status',
                 palette={'Alive': 'steelblue', 'Dead': 'tomato'},
                 plot_kws={'alpha': 0.4}, diag_kind='kde')
g.fig.suptitle('Pairplot — Variáveis Numéricas por Status', y=1.02, fontsize=14)
plt.show()

## 13. Conclusão

Esta análise exploratória revelou padrões consistentes no dataset de Câncer de Mama que são tanto clinicamente relevantes quanto úteis para a construção de modelos preditivos.

A variável-alvo (`Status`) apresenta **desbalanceamento de classes**, com pacientes vivos (Alive) em maioria. Qualquer modelo de classificação construído sobre esses dados deve levar isso em conta — por exemplo, via pesos por classe, sobreamostragem (SMOTE) ou ajuste de limiar de decisão — para evitar predições tendenciosas em favor da classe dominante.

Entre as variáveis numéricas, **Tumor Size**, **Regional Node Examined** e **Reginol Node Positive** apresentam as maiores correlações mútuas e a separação mais clara entre grupos de sobrevivência nos boxplots, sendo prováveis preditores de alta relevância. Os **Meses de Sobrevivência** seguem distribuições marcadamente distintas entre pacientes Alive e Dead, reforçando seu valor como variável de desfecho secundária ou como feature.

As variáveis de estadiamento clínico (T Stage e N Stage) confirmam o comportamento oncológico esperado: estágios mais avançados estão associados a tumores maiores e maior envolvimento linfonodal, com pacientes mortos (Dead) concentrados nos estágios superiores. O **status dos receptores hormonais** (Estrogênio e Progesterona) demonstra consistentemente que a expressão positiva dos receptores está associada a melhores desfechos de sobrevivência, em linha com as evidências clínicas estabelecidas.

Do ponto de vista de engenharia de features, o dataset se apresenta limpo (nenhum valor ausente detectado), porém algumas variáveis numéricas exibem distribuições assimétricas à direita e outliers notáveis — especialmente nas contagens relacionadas a linfonodos — que podem se beneficiar de transformação logarítmica ou escalonamento robusto antes da modelagem.

De forma geral, o dataset fornece uma base sólida para a construção de um modelo de classificação de sobrevivência, com features candidatos de alta relevância já identificados na análise.